# Tp 2 : Optimisation sans contrainte et descente de gradient

### Objectifs pédagogiques du TP
Dans cette séance de travaux pratiques, nous allons :
1. Construire un modèle mathématique d’un problème réel d’optimisation.
2. Étudier l’existence et l’unicité de la solution dans un problème sans contrainte.
3. Implémenter la descente de gradient :
    * Descente de gradient batch (BGD)
    * Descente de gradient stochastique simplifiée (SGD)
4. Observer l’influence du taux d’apprentissage et des paramètres du modèle.

### Table
- [1. Description du problème](#1)
- [2. Génération de la trajectoire initiale](#2)
- [3. Modélisation mathématique](#3)
    - [3.1 Fonction Objective](#3.1)
    - [3.2. Questions théoriques](#3.2)
    - [3.3. Implémentation de la fonction objectif](#3.3)
- [4. Approximation numérique du gradient](#4)
    - [4.1 Descente de gradient (BGD)](#4.1)
    - [4.2 Descente de gradient stochastique (SGD)](#4.2)
    - [4.3 Comparaison de BGD et SGD](#4.3)
- [5. Analyse expérimentale](#5)
- [6. Conclusion](#6)

<a name='1'></a>
### 1. Description du problème
On considère un problème de planification de trajectoire dans un espace tridimensionnel.
Un drone doit relier plusieurs points tout en :
* minimisant la longueur de la trajectoire ;
* gardant une trajectoire lisse ;
* évitant des obstacles sphériques.
La trajectoire est représentée par :
$$\mathbf{x} =[\mathbf{x}_1,\mathbf{x}_2,\dots,\mathbf{x}_N]^\top$$
où
$$\mathbf{x}_i=(x_i,y_i,z_i)$$
désigne le (i)-ème point de la trajectoire.

**Configuration des obstacles** : Supposons qu'il y ait plusieurs obstacles sphériques dans l'espace tridimensionnel, par exemple :
1. Obstacle 1 :
    - Centre $c_1 = (1,5,\, 1,5,\, 1,5)$
    - Rayon $r_1 = 0,5$
1. Obstacle 2 :
    - Centre $c_2 = (2,5,\, 2,5,\, 2,5)$
    - Rayon $r_2 = 0,5$

<a name='2'></a>

### 2. Génération de la trajectoire initiale
Nous générons aléatoirement $N$ points dans l'espace tridimensionnel comme trajectoire initiale, par exemple $N=3$ :
$$\mathbf{x}_{\text{init}} = \text{ points aléatoires dans } [0,3]^3$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import time

# Générer des points initiaux aléatoires
N = 10  # Nombre de points
np.random.seed(37)
x_init = np.random.rand(N, 3) * 3  # Points dans [0, 3]^3
x0 = x_init.flatten()

# Configuration des obstacles
obstacles = [
    {"center": np.array([1.5, 1.5, 1.5]), "radius": 0.5},
    {"center": np.array([2.5, 2.5, 2.5]), "radius": 0.5}
]

print(x_init)

<a name='3'></a>

### 3. Modélisation mathématique

<a name='3.1'></a>

#### 3.1 Fonction Objective

Nous définissons une fonction coût composée de trois parties :
1. Minimiser la longueur de la trajectoire (réduire la consommation d'énergie).
1. Minimiser la courbure de la trajectoire (réduire les changements d'accélération).
1. Éviter les collisions avec les obstacles (grâce à une forte pénalité).

**Fonction objective** :

Soit $x=(x_1,\\dots,x_N)$, avec $x_i\in\mathbb{R}^3$. On considère
$$
J(x)=\alpha\sum_{i=1}^{N-1}\|x_{i+1}-x_i\|
+\beta\sum_{i=2}^{N-1}\|x_{i+1}-2x_i+x_{i-1}\|^2
+\gamma\sum_{i=1}^{N}\sum_{j}\max\bigl(0,r_j-\|x_i-c_j\|\bigr)^2.
$$
Le premier terme mesure la longueur de la trajectoire, le deuxième pénalise les changements brusques de direction, et le troisième ajoute une pénalité quadratique lorsqu'un point de la trajectoire entre dans un obstacle sphérique.


<a name='3.2'></a>

#### 3.2. Questions théoriques
**Question 1 — Existence d’une solution**

Montrer que la fonction objectif est continue.

La fonction $x\mapsto \|x_{i+1}-x_i\|$ est continue car elle est composée d'applications linéaires et de la norme euclidienne. Le terme de courbure $x\mapsto \|x_{i+1}-2x_i+x_{i-1}\|^2$ est également continu. Enfin, pour chaque obstacle, $x\mapsto \max(0,r_j-\|x_i-c_j\|)^2$ est continu comme composition de fonctions continues. La somme finie de ces termes est donc continue.

**Question 2 — Existence d’un minimiseur**

Sur tout domaine fermé et borné, par exemple si l'on impose que les points restent dans $[0,3]^3$, la fonction $J$ est continue ; le théorème de Weierstrass garantit donc l'existence d'un minimiseur global. Numériquement, l'initialisation et la visualisation se font dans cette boîte bornée. Sans contrainte explicite de boîte, la longueur et le lissage ne rendent pas forcément le problème coercif dans toutes les directions, donc l'existence globale doit être discutée avec prudence.

**Question 3 — Unicité**

On ne peut pas garantir l'unicité en général. Le terme de collision avec obstacles rend le problème non convexe, et plusieurs trajectoires différentes peuvent contourner les obstacles avec des coûts comparables, par exemple par des côtés opposés. La descente de gradient peut donc converger vers des minima locaux différents selon l'initialisation et les paramètres.


<a name='3.3'></a>

#### 3.3. Implémentation de la fonction objectif

In [ ]:
def objective_function(x,
                       obstacles,
                       alpha=1.0,
                       beta=1.0,
                       gamma=1000):

    x = x.reshape(-1,3)

    N = len(x)

    # ---------------------------------------------------
    # 1. Longueur de trajectoire
    # ---------------------------------------------------

    path_length = 0.0

    for i in range(N-1):

        path_length += np.linalg.norm(x[i+1] - x[i])

    # ---------------------------------------------------
    # 2. Lissage
    # ---------------------------------------------------

    curvature = 0.0

    for i in range(1,N-1):

        curvature += np.linalg.norm(x[i+1] - 2*x[i] + x[i-1])**2

    # ---------------------------------------------------
    # 3. Collision
    # ---------------------------------------------------

    collision_penalty = 0.0

    for i in range(N):

        for obstacle in obstacles:
            distance = np.linalg.norm(x[i] - obstacle["center"])
            violation = max(0.0, obstacle["radius"] - distance)
            collision_penalty += violation**2

    # ---------------------------------------------------
    # Fonction finale
    # ---------------------------------------------------

    total_cost = (
        alpha * path_length
        + beta * curvature
        + gamma * collision_penalty
    )

    return total_cost


<a name='4'></a>

### 4. Approximation numérique du gradient
Dans ce TP, nous utilisons les différences finies.
$$\frac{\partial J}{\partial x_i} \approx \frac{J(x_i+\varepsilon)-J(x_i)}{\varepsilon}$$

In [ ]:
def gradient(x,
             obstacles,
             alpha=1.0,
             beta=1.0,
             gamma=1000.0,
             epsilon=1e-6):

    n = len(x)

    grad = np.zeros(n)

    f0 = objective_function(
        x,
        obstacles,
        alpha,
        beta,
        gamma
    )

    for i in range(n):

        x_perturbed = x.copy()

        x_perturbed[i] += epsilon

        f1 = objective_function(
            x_perturbed,
            obstacles,
            alpha,
            beta,
            gamma
        )

        grad[i] = (f1 - f0) / epsilon

    return grad

<a name='4.1'></a>

#### 4.1 Descente de gradient (BGD)
**Rappel théorique** :
La mise à jour s’effectue par :
$$x^{(k+1)} = x^{(k)} - \eta \nabla J(x^{(k)})$$
où :
* $\eta$ est le taux d’apprentissage ;
* $\nabla J$ est le gradient.

**Implémentation en Python** :

In [ ]:
def gradient_descent(
    x0,
    obstacles,
    alpha=1.0,
    beta=1.0,
    gamma=1000.0,
    learning_rate=0.01,
    max_iter=1000,
    tol=1e-6
):

    x = x0.copy()

    history = [x.copy()]

    for iteration in range(max_iter):

        grad = gradient(
            x,
            obstacles,
            alpha,
            beta,
            gamma
        )

        x_new = x - learning_rate * grad

        history.append(x_new.copy())

        if np.linalg.norm(x_new - x) < tol:

            print(
                f"Convergence atteinte à l'itération {iteration+1}"
            )

            break

        x = x_new

    return x.reshape(-1,3), history


<a name='4.2'></a>

#### 4.2 Descente de gradient stochastique (SGD)
Au lieu d’utiliser tous les points pour calculer le gradient, on choisit aléatoirement une petite partie.
Cela réduit le coût de calcul.

**Implémentation en Python** :

In [ ]:
def stochastic_gradient_descent(
    x0,
    obstacles,
    alpha=1.0,
    beta=1.0,
    gamma=1000.0,
    learning_rate=0.01,
    max_iter=1000
):

    x = x0.copy()

    n = len(x)

    history = [x.copy()]

    for iteration in range(max_iter):

        idx = np.random.randint(0,n) # choix aléatoire d'une coordonnée

        epsilon = 1e-6

        x_perturbed = x.copy()

        x_perturbed[idx] += epsilon

        f0 = objective_function(
            x,
            obstacles,
            alpha,
            beta,
            gamma
        )

        f1 = objective_function(
            x_perturbed,
            obstacles,
            alpha,
            beta,
            gamma
        )

        grad_i = (f1 - f0) / epsilon

        x[idx] = x[idx] - learning_rate * grad_i

        history.append(x.copy())

    return x.reshape(-1,3), history


<a name='4.3'></a>

#### 4.3 Comparaison de BGD et SGD

In [ ]:
x0 = x_init.flatten()

# -------------------------------
# Descente de gradient BGD
# -------------------------------

start_time = time.time()

x_bgd, history_bgd = gradient_descent(
    x0,
    obstacles
)

bgd_time = time.time() - start_time

# -------------------------------
# Descente de gradient SGD
# -------------------------------

start_time = time.time()

x_sgd, history_sgd = stochastic_gradient_descent(
    x0,
    obstacles
)

sgd_time = time.time() - start_time

In [ ]:
fig = plt.figure(figsize=(12,8))

ax = fig.add_subplot(111, projection='3d')

# Trajectoire initiale

ax.plot(
    x_init[:,0],
    x_init[:,1],
    x_init[:,2],
    'o-',
    label='Trajectoire initiale'
)

# BGD

ax.plot(
    x_bgd[:,0],
    x_bgd[:,1],
    x_bgd[:,2],
    'o-',
    label='BGD'
)

# SGD

ax.plot(
    x_sgd[:,0],
    x_sgd[:,1],
    x_sgd[:,2],
    'o-',
    label='SGD'
)

# Obstacles

for obstacle in obstacles:

    u, v = np.mgrid[
        0:2*np.pi:20j,
        0:np.pi:10j
    ]

    x = (
        obstacle["center"][0]
        + obstacle["radius"]
        * np.sin(u)
        * np.cos(v)
    )

    y = (
        obstacle["center"][1]
        + obstacle["radius"]
        * np.sin(u)
        * np.sin(v)
    )

    z = (
        obstacle["center"][2]
        + obstacle["radius"]
        * np.cos(u)
    )

    ax.plot_surface(
        x,
        y,
        z,
        color='r',
        alpha=0.3
    )

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")

ax.legend()

plt.title(
    "Optimisation de trajectoire"
)

plt.show()

<a name='5'></a>

### 5. Analyse expérimentale
**Question 4**
Comparer entre BGD et SGD :
- la vitesse de convergence ;
- la stabilité ;
- la qualité de la trajectoire finale.

**Réponse.** BGD calcule le gradient sur toutes les coordonnées à chaque itération : il est plus coûteux par itération, mais la direction est plus stable et la trajectoire finale est généralement plus régulière. SGD ne modifie qu'une coordonnée aléatoire à chaque étape : il est moins coûteux par itération, mais son évolution est plus bruitée et la qualité finale dépend davantage du hasard et du taux d'apprentissage.

**Question 5**
Modifier le taux d’apprentissage :
$$\eta = 0.1,\quad 0.01,\quad 0.001$$
Que se passe-t-il ?

**Réponse.** Pour $\eta=0.001$, la convergence est lente mais stable. Pour $\eta=0.01$, on obtient souvent un bon compromis entre vitesse et stabilité. Pour $\eta=0.1$, les mises à jour peuvent devenir trop grandes : la méthode peut osciller, créer une trajectoire irrégulière, voire augmenter la fonction coût.

**Question 6**
Modifier les paramètres et interpréter les résultats :
- $lpha$
- $eta$
- $\gamma$

**Réponse.** $lpha$ augmente l'importance de la longueur totale : une grande valeur favorise une trajectoire courte. $eta$ augmente l'importance du lissage : une grande valeur réduit les changements brusques de direction. $\gamma$ augmente la pénalité de collision : une grande valeur force davantage l'évitement des obstacles, mais peut rendre l'optimisation plus raide et plus difficile numériquement.

<a name='6'></a>

### 6. Conclusion

Dans ce TP, nous avons construit une fonction objectif pour un problème de trajectoire 3D, vérifié la continuité et discuté l'existence et l'unicité. Les expériences montrent que BGD est plus stable, tandis que SGD est plus bruité. Le choix du learning rate et des poids $lpha,eta,\gamma$ est déterminant pour obtenir une trajectoire courte, lisse et sans collision.
